In [1]:
%load_ext autoreload
%autoreload 2
import jax
jax.config.update('jax_platform_name', 'cpu')
jax.config.update('jax_platforms', 'cpu')
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp

import numpy as np
from datetime import datetime, timezone

import atmo3 as a3

import pycraf
from astropy import units as u

from scipy import constants

import matplotlib.pyplot as plt
import numpy as np

# 1. Update global parameters for presentation visibility
plt.rcParams.update({
    'font.size': 18,              # General font size for text
    'axes.labelsize': 20,         # Size of X and Y axis labels
    'axes.titlesize': 24,         # Size of the graph title
    'xtick.labelsize': 16,        # Size of the numbers on the X axis
    'ytick.labelsize': 16,        # Size of the numbers on the Y axis
    'legend.fontsize': 16,        # Size of the legend text
    'lines.linewidth': 3,       # Thicker lines for the plotted data
})

In [2]:
# ==========================================
# --- 1. Grid & Simulation Setup ---
# ==========================================
nside_grid = [128, 18, 1024]
box_length = [5000., 5000., 50000.]

freqs = jnp.array([27, 39, 145, 225, 280]) # in GHz, SO frequencies

site_altitude = 5100.
site_coordinates = [-67.78, -22.95] 
time_utc = datetime(2023, 9, 16, 0, 0, tzinfo=timezone.utc)

atmo3_data = '/pscratch/sd/s/shamikg/atmo3_data/'

geopotfile = f'{atmo3_data}era5/2023/geopt.202309.ap1e5.291.0_293.0_-24.0_-22.0_025.nc'
tempfile   = f'{atmo3_data}era5/2023/ta.202309.ap1e5.291.0_293.0_-24.0_-22.0_025.nc'
spechfile  = f'{atmo3_data}era5/2023/q.202309.ap1e5.291.0_293.0_-24.0_-22.0_025.nc'
o3file     = f'{atmo3_data}era5/2023/o3.202309.ap1e5.291.0_293.0_-24.0_-22.0_025.nc'
apexfile   = f'{atmo3_data}apex/meteo_apex_2006_2025.csv'

# Initialize Atmosphere
atmo_box = a3.Atmosphere(
    nside_grid=nside_grid,
    box_length_in_m=box_length,
    site_altitude=site_altitude,
    site_coordinates=site_coordinates,
    time_utc=time_utc,
    geopotential_file_era5=geopotfile,
    temperature_file_era5=tempfile,
    spec_humidity_file_era5=spechfile,
    apex_datafile=apexfile
)

# Extract Thermodynamic Profiles
ds = atmo_box.grid_wsp.grid_spacing[2] * jnp.ones((nside_grid[2],))
T = atmo_box.atm_calibrator.temperature_profile
P = atmo_box.atm_calibrator.pressure

# Base water vapor profile
rho_w_base = atmo_box.atm_calibrator.q2rho_h2o * atmo_box.atm_calibrator.spec_humidity_profile

# Ozone Setup
o3_profile_1d = atmo_box.super_grid.era5_interp2site(o3file)
M_air = 28.9647e-3 
M_o3 = 47.9982e-3  
vmr_o3 = o3_profile_1d * (M_air / M_o3)
P_o3 = vmr_o3 * P 

# Vectorized attenuation functions
vmap_attenuation = jax.vmap(a3.compute_attenuation_water_vapor_am, in_axes=(0, 0, 0, None), out_axes=1)
vmap_attenuation_dry = jax.vmap(a3.compute_attenuation_dry_air_am, in_axes=(0, 0, 0, None), out_axes=1)
vmap_attenuation_o3 = jax.vmap(a3.compute_attenuation_ozone_am, in_axes=(0, 0, 0, 0, None), out_axes=1)

gamma_wet = vmap_attenuation(T, P, rho_w_base, freqs)
gamma_dry = vmap_attenuation_dry(T, P, rho_w_base, freqs)
gamma_o3  = vmap_attenuation_o3(T, P, P_o3, rho_w_base, freqs)

# Total Attenuation and Transmission
total_attenuation = gamma_wet + gamma_dry + gamma_o3 #(Nf, Nz)


d_tau = total_attenuation * (ds[None, :] / 1000.0) * (jnp.log(10.0) / 10.0)

# 3. Integrate vertically (cumulative sum from ground z=0 up to layer z)
# Shape remains (Nf, Nz)
tau_zenith = jnp.cumsum(d_tau, axis=1) 

1.1099999999999999 0.0607453701939497


In [ ]:
#We need to generate the clouds